In [1]:
# Imports

import os 
import pickle 
import numpy as np 
import pandas as pd 
from datetime import datetime 
from pathlib import Path 

from sklearn.model_selection import train_test_split , RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor , GradientBoostingRegressor , HistGradientBoostingRegressor
from sklearn.linear_model import Ridge , LinearRegression
from sklearn.svm import SVR 
from xgboost import XGBRegressor
# from sklearn.pipeline import Pipeline 
from sklearn.metrics import mean_squared_error , mean_absolute_error,r2_score

import mlflow 
import mlflow.sklearn 




In [2]:
# Paths Configurations 

data_path = Path('data/flights_cleaned.npz')

artifacts_dir = Path('artifacts')

mlflow_experiment_name = 'flight_price_baseline'
random_state = 42 

mlflow.set_experiment(mlflow_experiment_name)



d:\Deep Learning Projects\travel-mlops-capstone\travel_venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)
Traceback (most recent call last):
  File "d:\Deep Learning Projects\travel-mlops-capstone\travel_venv\lib\site-packages\mlflow\store\tracking\file_store.py", line 378, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "d:\Deep Learning Projects\travel-mlops-capstone\travel_venv\lib\site-packages\mlflow\store\tracking\file_store.py", line 476, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "d:\Deep Learning Projects\travel-m

<Experiment: artifact_location='file:///opt/airflow/mlruns/568030847319124871', creation_time=1766584564277, experiment_id='568030847319124871', last_update_time=1766584564277, lifecycle_stage='active', name='flight_price_baseline', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [3]:
# load data 

assert data_path.exists(),f"{data_path} not found"
npz = np.load(data_path , allow_pickle = True)
if "X" in npz.files and 'y' in npz.files:
    X = npz["X"]
    y = npz['y']

else:
    raise ValueError("Could not find X and y arrays inside flights_cleaned.npz")

print("X shape :", X.shape )
print("Y shape :",y.shape)




X shape : (271888, 34)
Y shape : (271888,)


In [4]:
# load feature names
with open(
    artifacts_dir/"feature_names.pkl",
    "rb"
) as f:

    feature_names = pickle.load(f)

In [5]:
# Train test split 

xtrain , xtest , ytrain , ytest = train_test_split(X , y , test_size = 0.1 , random_state= random_state)

print('Train shapes:', xtrain.shape, ytrain.shape)
print('Test shapes :', xtest.shape, ytest.shape)

Train shapes: (244699, 34) (244699,)
Test shapes : (27189, 34) (27189,)


In [6]:
import json 
from sklearn.base import clone

In [7]:
def evaluate_and_log(model_name , model , xtrain , ytrain , xtest , ytest , param_search = None , n_iter = 20):

    """
    Train(with optional RandomizedSearchCV) and log everything MLFLOW

    Returns : best_estimator , dict(metrics) 
    
   
    """
    run_name = f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    with mlflow.start_run(run_name = run_name):
        mlflow.set_tag('model_type' , model_name)

        # If param_search provided ,run RandomizedSearchCV 

        if param_search:
            search = RandomizedSearchCV(
                estimator = clone(model),
                param_distributions=param_search,
                n_iter = min(n_iter , 40),
                cv = 3,
                scoring = 'neg_root_mean_squared_error',
                random_state = random_state,
                n_jobs = -1,
                verbose = 0
            )

            search.fit(xtrain , ytrain)
            best = search.best_estimator_
            best_params = search.best_params_
            mlflow.log_params({f"best_{k}": v for k , v in best_params.items()})

            # log Cross Validation results

            try:
                cvres = pd.DataFrame(search.cv_results_)
                cv_summary = cvres[['params','mean_test_score','std_test_score','rank_test_score']].to_dict(orient = 'records')

                mlflow.log_text(json.dumps(cv_summary , default = str),'cv_summary.json')

            except Exception as e:
                print(e)
                pass

        else:
            best = clone(model)
            best.fit(xtrain , ytrain)
            mlflow.log_params({'note':'no-hyperparam-search'})
        

        # predict and metrics

        preds = best.predict(xtest)
        rmse = mean_squared_error(ytest , preds , squared=False)
        mae = mean_absolute_error(ytest , preds) 
        r2 = r2_score(ytest , preds)

        mlflow.log_metric('rmse', float(rmse))
        mlflow.log_metric('mae', float(mae))
        mlflow.log_metric('r2', float(r2))

        # log the model via mlflow.sklearn

        mlflow.sklearn.log_model(best , artifact_path = 'model' )

        # save and log a pickle locally as well

        model_file = artifacts_dir/f'{model_name}_best.pkl'
        with open(model_file , 'wb') as f:
            pickle.dump(best , f)

        mlflow.log_artifact(str(model_file) , artifact_path='model_files')

        # if model has feature_importances_ , log top features 

        try:
            if hasattr(best, "feature_importances_"):
    
                importances = best.feature_importances_

            elif hasattr(best, "coef_"):

                importances = np.abs(best.coef_)

            else:

                importances = None
            
            if importances is not None:
                imp_df = pd.DataFrame({"feature_name":feature_names,"importance":importances})
                imp_df = imp_df.sort_values(by='importance' , ascending = False)
                imp_csv = artifacts_dir/f'{model_name}_feature_importances.csv'
                imp_df.to_csv(imp_csv , index = False)

                mlflow.log_artifact(str(imp_csv) , artifact_path = 'feature_importances')

        except Exception :
            pass



        print(f"{model_name} done - RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

        return best , {'rmse':rmse , 'mae':mae  , 'r2':r2}
    
    

In [8]:
# Defining models and search spaces

models_and_search = []

# 1) Linear Regression

linear = LinearRegression()

linear_search = None

models_and_search.append(
    (
        'linear_regression',
        linear,
        linear_search
    )
)


# 2) Ridge Regression

ridge = Ridge(
    random_state=random_state
)

ridge_search = {

    'alpha': [
        0.1,
        1.0,
        10.0,
        100.0
    ]

}

models_and_search.append(

    (
        'ridge',
        ridge,
        ridge_search
    )

)


# 3) Support Vector Regressor

# svr = SVR(
#     kernel='rbf',
#     gamma='scale',
#     C=1.0
# )


# svr_search = None

# models_and_search.append(

#     (
#         'svr',
#         svr,
#         svr_search
#     )

# )


# 4) Random Forest

rf = RandomForestRegressor(
    random_state=random_state
)

rf_search = {

    'n_estimators': [
        100,
        200,
        400
    ],

    'max_depth': [
        6,
        10,
        20,
        None
    ],

    'min_samples_leaf': [
        1,
        2,
        4
    ]

}

models_and_search.append(

    (
        'random_forest',
        rf,
        rf_search
    )

)


# 5) Gradient Boosting

gb = GradientBoostingRegressor(
    random_state=random_state
)

gb_search = {

    'n_estimators': [
        100,
        200,
        400
    ],

    'learning_rate': [
        0.01,
        0.05,
        0.1
    ],

    'max_depth': [
        3,
        5,
        8
    ]

}

models_and_search.append(

    (
        'grad_boost',
        gb,
        gb_search
    )

)


# 6) Histogram Gradient Boosting

hgb = HistGradientBoostingRegressor(
    random_state=random_state
)

hgb_search = {

    'max_iter': [
        100,
        200,
        400
    ],

    'learning_rate': [
        0.01,
        0.05,
        0.1
    ],

    'max_depth': [
        3,
        6,
        12
    ]

}

models_and_search.append(

    (
        'hist_gb',
        hgb,
        hgb_search
    )

)


# 7) XGBoost

xgb = XGBRegressor(

    objective='reg:squarederror',

    random_state=random_state,

    n_jobs=-1

)

xgb_search = {

    'n_estimators': [
        100,
        200,
        400
    ],

    'learning_rate': [
        0.01,
        0.05,
        0.1
    ],

    'max_depth': [
        3,
        6,
        10
    ]

}

models_and_search.append(

    (
        'xgboost',
        xgb,
        xgb_search
    )

)

In [9]:
# Running training loop

results = {}

for name , model , search_space in models_and_search:
    print('\nStarting:',name)
    best_est , metrics = evaluate_and_log(
        model_name = name ,
        model = model , 
        xtrain=xtrain ,
        ytrain = ytrain,
        xtest = xtest , 
        ytest = ytest,
        param_search = search_space,
        n_iter = 20
    )
    results[name] = {'estimator':best_est , 'metrics':metrics}

# save summary of results

summary_df = pd.DataFrame([{
    'model':k,
    'rmse':v['metrics']['rmse'],
    'mae':v['metrics']['mae'],
    'r2':v['metrics']['r2']
} for k , v in results.items()])

summary_df = summary_df.sort_values('rmse')
summary_df.to_csv(artifacts_dir/'model_comparison.csv' , index = False)

mlflow.log_artifacts(str(artifacts_dir/'model_comparison.csv'))
print('\nAll models trained. Summary:')
print(summary_df)




Starting: linear_regression


2026/06/06 21:43:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


linear_regression done - RMSE: 103.0139, MAE: 80.7848, R2: 0.9190

Starting: ridge


d:\Deep Learning Projects\travel-mlops-capstone\travel_venv\lib\site-packages\sklearn\model_selection\_search.py:307: UserWarning: The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
2026/06/06 21:44:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


ridge done - RMSE: 103.0136, MAE: 80.7764, R2: 0.9190

Starting: random_forest


...

grad_boost done - RMSE: 0.1803, MAE: 0.1238, R2: 1.0000

Starting: hist_gb


2026/06/06 23:55:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


hist_gb done - RMSE: 3.1842, MAE: 2.3688, R2: 0.9999

Starting: xgboost


2026/06/07 00:13:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgboost done - RMSE: 0.0072, MAE: 0.0020, R2: 1.0000

All models trained. Summary:
               model        rmse        mae        r2
5            xgboost    0.007187   0.001981  1.000000
3         grad_boost    0.180290   0.123751  1.000000
2      random_forest    0.550116   0.043950  0.999998
4            hist_gb    3.184231   2.368757  0.999923
1              ridge  103.013649  80.776368  0.918952
0  linear_regression  103.013926  80.784821  0.918951
